# Production Patterns for LangChain

Notebooks **02–16** built the pieces: LCEL chains, RAG, tools, agents, memory, observability, and resilience. **Production** is where those pieces meet your application — FastAPI routes, configuration, service boundaries, testing, deployment, and operational discipline.

This capstone notebook assembles a **Notes API assistant** service layer: factory-built chains, async HTTP handlers (**07**), session memory (**14**), traced invocations (**15**), fallback-ready models (**16**), and patterns you can lift into a real repo without coupling business logic to LangChain imports in every file.

Prerequisites: the full **01. LangChain/** track (**02–16**). This is the final notebook in the series.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"
LANGCHAIN_API_KEY = "ls-placeholder-replace-with-your-langsmith-key"  # optional

import json
import os
from dataclasses import dataclass
from typing import AsyncIterator

import langchain
import langchain_core

print("langchain:", langchain.__version__)
print("langchain-core:", langchain_core.__version__)

---

## 1. Production Architecture

Separate **HTTP concerns** from **LangChain orchestration**:

```
┌─────────────────────────────────────────────────────────────┐
│  FastAPI / worker / CLI                                     │
│    auth · rate limits · request validation · HTTP errors      │
└───────────────────────────┬─────────────────────────────────┘
                            │
┌───────────────────────────▼─────────────────────────────────┐
│  Service layer (your code)                                  │
│    ChatService · RagService · AgentService                  │
└───────────────────────────┬─────────────────────────────────┘
                            │
┌───────────────────────────▼─────────────────────────────────┐
│  Chain factory (startup)                                    │
│    build_qa_chain() · build_rag_chain() · build_agent()     │
└───────────────────────────┬─────────────────────────────────┘
                            │
     ┌──────────────────────┼──────────────────────┐
     ▼                      ▼                      ▼
  ChatOpenAI            Chroma retriever      create_agent
  prompts/parsers       embeddings            tools + checkpointer
└─────────────────────────────────────────────────────────────┘
                            │
                            ▼
              LangSmith / logs / metrics (15)
```

| Layer | Owns | Does not own |
|-------|------|--------------|
| **Route** | HTTP status, auth, DTOs | Prompt wording |
| **Service** | `invoke`/`astream`, session ids | SQL schema design |
| **Factory** | Runnable graph wiring | Per-request user text |
| **LangChain** | Model calls, retrieval | Business authorization |

---

## 2. Configuration — Settings and Secrets

Load secrets and model defaults from the environment — never hardcode keys in chain modules:

In [ ]:
from pydantic_settings import BaseSettings


class AppSettings(BaseSettings):
    openai_api_key: str = OPENAI_API_KEY
    default_model: str = "gpt-4o-mini"
    premium_model: str = "gpt-4o"
    langchain_tracing: bool = False
    langchain_project: str = "notes-api-assistant"
    chroma_persist_dir: str = "./data/chroma_notes"

    model_config = {"env_prefix": "APP_"}


settings = AppSettings()
print("model:", settings.default_model)
print("tracing:", settings.langchain_tracing)

Enable LangSmith in deployment:

```bash
export LANGCHAIN_TRACING_V2=true
export LANGCHAIN_API_KEY=ls__...
export LANGCHAIN_PROJECT=notes-api-prod
```

Keep **`OPENAI_API_KEY`** in secrets manager (Vault, AWS SM) — inject at container start.

---

## 3. Chain Factory — Build Once at Startup

Construct Runnables when the app boots — not per request:

In [ ]:
import uuid

from langchain.agents import create_agent
from langchain_chroma import Chroma
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import ConfigurableField, RunnableConfig, RunnableParallel, RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.checkpoint.memory import MemorySaver


CORPUS = [
    Document(page_content="The Notes API is built with FastAPI. Routes live under /notes.", metadata={"id": "c1"}),
    Document(page_content="Run alembic upgrade head after pulling database migrations.", metadata={"id": "c2"}),
    Document(page_content="JWT bearer tokens use Authorization: Bearer header.", metadata={"id": "c3"}),
    Document(page_content="Pytest DB fixtures belong in conftest.py.", metadata={"id": "c4"}),
]


def format_docs(docs: list[Document]) -> str:
    if not docs:
        return "(no relevant context found)"
    return "\n\n".join(f"[{d.metadata.get('id', '?')}] {d.page_content}" for d in docs)


def build_chain_factory(cfg: AppSettings):
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=cfg.openai_api_key)
    economy_llm = ChatOpenAI(model=cfg.default_model, api_key=cfg.openai_api_key, temperature=0)
    premium_llm = ChatOpenAI(model=cfg.premium_model, api_key=cfg.openai_api_key, temperature=0)

    llm = economy_llm.configurable_alternatives(
        ConfigurableField(id="llm"),
        default_key="economy",
        alternatives={"economy": economy_llm, "premium": premium_llm},
    )

    qa_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a concise backend engineering assistant."),
        ("human", "{question}"),
    ])
    qa_chain = (
        qa_prompt | llm | StrOutputParser()
    ).with_config({"run_name": "qa", "tags": ["notes-api"]})

    vectorstore = Chroma.from_documents(
        documents=CORPUS,
        embedding=embeddings,
        collection_name=f"notes_prod_{uuid.uuid4().hex[:8]}",
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    rag_prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer using ONLY the context. If unknown, say I don't know.\n\n{context}"),
        ("human", "{question}"),
    ])
    rag_chain = (
        RunnableParallel(context=retriever | format_docs, question=RunnablePassthrough())
        | rag_prompt | llm | StrOutputParser()
    ).with_config({"run_name": "rag", "tags": ["notes-api", "rag"]})

    chat_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder("history"),
        ("human", "{input}"),
    ])
    chat_chain = chat_prompt | llm | StrOutputParser()

    session_store: dict[str, InMemoryChatMessageHistory] = {}

    def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
        if session_id not in session_store:
            session_store[session_id] = InMemoryChatMessageHistory()
        return session_store[session_id]

    chat_with_memory = RunnableWithMessageHistory(
        chat_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history",
    )

    @tool
    def search_docs(query: str) -> str:
        """Search internal documentation chunks."""
        return format_docs(retriever.invoke(query))

    agent = create_agent(
        model=llm,
        tools=[search_docs],
        system_prompt="Use search_docs for documentation questions.",
        checkpointer=MemorySaver(),
    )

    return {
        "qa": qa_chain,
        "rag": rag_chain,
        "chat": chat_with_memory,
        "agent": agent,
    }


chains = build_chain_factory(settings)
print("built:", list(chains.keys()))

Factory returns a **dict of named runnables** — routes pick the right entry without re-importing LangChain everywhere.

---

## 4. Service Layer — Hide LangChain from Routes

Routes call **services**; services call **chains**:

In [ ]:
@dataclass
class RagResponse:
    answer: str
    context: str | None = None


class NotesAssistantService:
    def __init__(self, chain_map: dict):
        self._chains = chain_map

    def _base_config(self, user_tier: str = "free") -> RunnableConfig:
        llm_key = "premium" if user_tier == "pro" else "economy"
        return RunnableConfig(
            configurable={"llm": llm_key},
            metadata={"user_tier": user_tier},
            tags=["api", user_tier],
        )

    def ask(self, question: str, user_tier: str = "free") -> str:
        return self._chains["qa"].invoke({"question": question}, config=self._base_config(user_tier))

    def ask_docs(self, question: str, user_tier: str = "free") -> RagResponse:
        answer = self._chains["rag"].invoke(question, config=self._base_config(user_tier))
        return RagResponse(answer=answer)

    def chat(self, session_id: str, text: str, user_tier: str = "free") -> str:
        config = self._base_config(user_tier)
        config["configurable"]["session_id"] = session_id
        return self._chains["chat"].invoke({"input": text}, config=config)

    def agent_turn(self, thread_id: str, text: str, user_tier: str = "free") -> str:
        config = self._base_config(user_tier)
        config["configurable"]["thread_id"] = thread_id
        result = self._chains["agent"].invoke(
            {"messages": [HumanMessage(content=text)]},
            config=config,
        )
        for msg in reversed(result["messages"]):
            if getattr(msg, "content", None):
                return msg.content if isinstance(msg.content, str) else str(msg.content)
        return ""

    async def astream_qa(self, question: str, user_tier: str = "free") -> AsyncIterator[str]:
        async for chunk in self._chains["qa"].astream(
            {"question": question}, config=self._base_config(user_tier)
        ):
            yield chunk


service = NotesAssistantService(chains)
print(service.ask("What is JWT?"))

The service layer is your **swap point** — if you outgrow LangChain, rewrite the factory and service internals while keeping HTTP contracts stable.

---

## 5. FastAPI Routes — Thin Handlers

Conceptual route module — validate input, delegate to service, map errors:

In [ ]:
FASTAPI_ROUTES = '''
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

app = FastAPI(title="Notes API Assistant")

# Built at startup (see build_chain_factory)
service = NotesAssistantService(chains)


class AskRequest(BaseModel):
    question: str
    user_tier: str = "free"


class ChatRequest(BaseModel):
    session_id: str
    message: str
    user_tier: str = "free"


@app.post("/ask")
def ask(req: AskRequest):
    try:
        return {"answer": service.ask(req.question, req.user_tier)}
    except Exception as exc:
        raise HTTPException(status_code=502, detail="LLM provider error") from exc


@app.post("/rag")
def rag(req: AskRequest):
    result = service.ask_docs(req.question, req.user_tier)
    return {"answer": result.answer}


@app.post("/chat")
def chat(req: ChatRequest):
    return {"reply": service.chat(req.session_id, req.message, req.user_tier)}


@app.post("/chat/stream")
async def chat_stream(req: AskRequest):
    async def tokens():
        async for chunk in service.astream_qa(req.question, req.user_tier):
            yield chunk
    return StreamingResponse(tokens(), media_type="text/plain")
'''
print(FASTAPI_ROUTES[:500], "...")

Patterns from **07** (async/streaming) and **14** (session ids). Add **auth middleware** before exposing agent or chat routes.

---

## 6. Application Lifecycle

```
startup
  ├─ load AppSettings from env
  ├─ optional: enable LANGCHAIN_TRACING_V2
  ├─ chains = build_chain_factory(settings)
  ├─ service = NotesAssistantService(chains)
  └─ warm up: optional index health check / embed probe

per request
  ├─ auth + rate limit
  ├─ service.method(..., config with tags/metadata)
  └─ return DTO (never raw AIMessage)

shutdown
  └─ flush traces / close vector store clients
```

Use FastAPI **`lifespan`** context manager for startup/shutdown hooks.

---

## 7. Resilience in Production

Stack patterns from **16** at factory time:

In [ ]:
from langchain_core.runnables import RunnableLambda


def build_resilient_qa(cfg: AppSettings):
    llm = ChatOpenAI(model=cfg.default_model, api_key=cfg.openai_api_key, temperature=0)
    backup = ChatOpenAI(model=cfg.default_model, api_key=cfg.openai_api_key, temperature=0.2)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are helpful."),
        ("human", "{question}"),
    ])
    return (
        prompt | llm.with_fallbacks([backup]) | StrOutputParser()
    ).with_retry(stop_after_attempt=2).with_fallbacks([
        RunnableLambda(lambda _: "Assistant temporarily unavailable."),
    ])


resilient = build_resilient_qa(settings)
print(resilient.invoke({"question": "What is Alembic?"}))

Log when fallbacks activate (**15**) — silent degradation hides provider outages.

---

## 8. Prompt Versioning and Serialization

Store prompts in git as serialized JSON (**04**):

In [ ]:
from langchain_core.load import dumpd, loads

versioned_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are the Notes API assistant v2. Cite doc ids when possible."),
    ("human", "{question}"),
])

serialized = dumpd(versioned_prompt)
restored = loads(serialized)

print("round-trip type:", type(restored).__name__)
print("keys in dump:", list(serialized.keys())[:4], "...")

Tag traces with **`metadata.prompt_version`** — compare quality in LangSmith when prompts change.

---

## 9. Testing Strategy

| Layer | Test with | Validates |
|-------|-----------|----------|
| **Tools** | Fake `AIMessage.tool_calls` (**12**) | Executor wiring |
| **Chains** | `FakeListChatModel` (**03**) | Prompt + parser |
| **Service** | Mock chain map | HTTP DTO contracts |
| **RAG** | Fixed corpus + golden queries | Retrieval ids |
| **Integration** | Staging + real API | End-to-end quality |

CI should **not** require live OpenAI for unit tests.

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

fake_llm = FakeListChatModel(responses=["JWT uses the Authorization Bearer header."])
test_prompt = ChatPromptTemplate.from_template("Answer: {question}")
test_chain = test_prompt | fake_llm | StrOutputParser()

assert "Bearer" in test_chain.invoke({"question": "JWT header?"})
print("offline chain test passed")

---

## 10. Evaluation Smoke Set

Maintain a small golden set — run on every prompt or model change:

In [ ]:
GOLDEN_QUERIES = [
    {"question": "Alembic upgrade command?", "expect_substr": "alembic"},
    {"question": "JWT header name?", "expect_substr": "Authorization"},
    {"question": "pytest DB fixtures file?", "expect_substr": "conftest"},
]


def run_smoke_eval(chain, queries: list[dict]) -> list[dict]:
    results = []
    for row in queries:
        answer = chain.invoke(row["question"]).lower()
        ok = row["expect_substr"].lower() in answer
        results.append({**row, "pass": ok, "answer_preview": answer[:80]})
    return results


# smoke = run_smoke_eval(chains["rag"], GOLDEN_QUERIES)
# print(json.dumps(smoke, indent=2))
print(f"golden set: {len(GOLDEN_QUERIES)} queries (uncomment to run live eval)")

Upload the same dataset to **LangSmith** for tracked regressions over time.

---

## 11. Security and Compliance

| Risk | Mitigation |
|------|------------|
| **API key leakage** | Env/secrets manager; never log keys |
| **PII in traces** | LangSmith masking; log ids not full text (**15**) |
| **Prompt injection** | System rules + tool sandboxing; validate outputs |
| **Agent side effects** | HITL middleware (**13**); idempotent tools (**12**) |
| **Cross-tenant data** | Separate `session_id` / `thread_id` per tenant (**14**) |
| **Over-broad retrieval** | Metadata filters on retriever (**10**) |

---

## 12. Cost and Performance Controls

| Lever | Mechanism |
|-------|----------|
| **Model tier** | `configurable_alternatives` economy vs premium (**16**) |
| **Retrieval k** | Lower `k` for latency-sensitive routes |
| **Caching** | `InMemoryCache` dev only; careful in prod (**15**) |
| **Agent limits** | `ModelCallLimitMiddleware` (**13**) |
| **Streaming** | Better UX without extra model calls (**07**) |
| **Batch eval** | `max_concurrency` on offline jobs (**07**) |
| **Context trim** | `trim_messages` / summarization (**14**) |

---

## 13. Choosing the Right Pattern

| User need | Ship | Notebooks |
|-----------|------|----------|
| FAQ over fixed docs | **RAG chain** | **08–11** |
| Open-ended chat | **Chat + memory** | **03–04**, **14** |
| Optional doc lookup + tools | **Agent** | **12–13** |
| Structured API output | **Parser / `response_format`** | **05**, **13** |
| Classify then route | **RunnableBranch** | **06** |

Most production APIs expose **one primary pattern** per endpoint — avoid sending every request through a maximal agent.

---

## 14. Deployment Checklist

| Category | Check |
|----------|-------|
| **Dependencies** | Pin `langchain`, `langchain-core`, provider packages |
| **Secrets** | `OPENAI_API_KEY`, optional `LANGCHAIN_API_KEY` |
| **Tracing** | LangSmith enabled in staging/prod |
| **Health** | `/health` + dependency probe (embed API reachable) |
| **Timeouts** | HTTP client + LLM request timeouts |
| **Fallbacks** | Backup model or static outage message (**16**) |
| **Memory store** | Durable checkpointer if using agents (**14**) |
| **Vector index** | Persisted Chroma/pgvector; versioned ingest (**09**) |
| **Eval** | Golden set passes before promote |
| **Runbook** | Disable agent tools, switch to RAG-only degrade mode |

---

## 15. Anti-Patterns to Avoid

| Anti-pattern | Why it hurts |
|--------------|-------------|
| LangChain calls in every route file | Untestable, tangled imports |
| Building chains per request | Slow; leaks connections |
| Agent for every endpoint | Cost, latency, unpredictability |
| No tracing until prod incident | Multi-step failures are opaque |
| Returning raw `AIMessage` to clients | Leaks tool metadata |
| `MemorySaver` in production | Lost sessions on restart |
| Skipping eval on prompt edits | Silent quality regression |
| Monolithic 500-line notebook in prod | Use modules + factory + service |

---

## 16. Full Track Recap

```
01 Introduction ──► why LangChain, ecosystem, learning path
02 Runnable/LCEL ──► invoke, stream, pipe, RunnableConfig
03 Chat Models ──► messages, ChatOpenAI, streaming
04 Prompts ──► ChatPromptTemplate, placeholders, versioning
05 Parsers ──► structured output, validation
06 Composition ──► parallel, branch, RAG dict pattern
07 Streaming/async ──► batch, astream, FastAPI sketch
08 Documents ──► chunking, metadata
09 Embeddings ──► Chroma, indexing
10 Retrievers ──► as_retriever, MMR, format_docs
11 RAG ──► end-to-end LCEL, conversational RAG
12 Tools ──► @tool, bind_tools, manual loop
13 Agents ──► create_agent, middleware, checkpointing
14 Memory ──► RunnableWithMessageHistory, trim, threads
15 Observability ──► callbacks, cache, LangSmith
16 Resilience ──► fallbacks, retry, configurable
17 Production ──► factory, service, deploy (this notebook)
```

You now have the vocabulary to read LangChain docs, debug traces, and ship maintainable LLM features behind your own API boundary.

---

## 17. Summary

**Production LangChain** means a **factory** that builds chains once, a **service layer** that your routes call, and **operational wrappers** — config, tracing, fallbacks, tests, and eval — applied consistently across Q&A, RAG, chat, and agent flows.

Key takeaways:

- **Thin routes, thick services, single factory** — the core architectural rule.
- **Settings** load secrets; **RunnableConfig** carries per-request tier and trace metadata.
- **Streaming** and **async** map directly to FastAPI (`astream`, `StreamingResponse`).
- **Serialize prompts**; version and eval them on every change.
- **Test offline** with fakes; reserve live API calls for integration/smoke eval.
- **Pick the simplest pattern** (RAG vs chat vs agent) per endpoint.
- **Pin dependencies**, enable tracing, and plan degrade modes before launch.

Demonstrations built a chain factory, wrapped it in `NotesAssistantService`, sketched FastAPI routes, applied resilience stacks, serialized prompts, outlined testing and eval, and recapped the full **01. LangChain/** track.

This completes the LangChain series. For multi-framework context, see **00. Orchestration Frameworks/**; for explicit graph authoring beyond `create_agent`, explore **LangGraph** in that section.